# P125 — LayoutLM: preentrenamiento de texto y disposición para comprensión de documentos

## 1. Título y paper

**Paper:** *LayoutLM: Pre-training of Text and Layout for Document Image Understanding*  
**Autoría:** Yiheng Xu, Minghao Li, Lei Cui, Shaohan Huang, Furu Wei, Ming Zhou  
**Año y venue:** 2020 · KDD 2020, 1192–1200  
**Nivel:** L2 · **Motor:** `layoutlm`  
**Ficha completa:** [`P125_layoutlm`](../../papers/foundational/P125_layoutlm/README.md)

**Hito:** Añade la posición en la página como una incrustación más, y con eso convierte un modelo de lenguaje en un lector de formularios y facturas.

- [doi:10.1145/3394486.3403172](https://doi.org/10.1145/3394486.3403172)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un documento no es una secuencia de texto: es texto colocado. Al linealizar una factura de dos columnas, el OCR intercala campos que no se relacionan, y un modelo que solo ve la cadena no puede emparejar cada etiqueta con su valor.
2. Ejecutar una implementación mínima de la propuesta: Preentrenar sobre millones de documentos escaneados un modelo que recibe, para cada token, su texto y las coordenadas de su caja delimitadora, con objetivos de enmascarado que obligan a usar las dos señales.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P09


## 4. Intuición

El OCR entrega una factura de dos columnas como una sola cadena leída por filas. El texto está perfectamente reconocido y aun así «Factura n.º» acaba emparejado con «Fecha».


## 5. Concepto mínimo

```text
Secuencia plana : token → embedding(texto) + embedding(posición en la SECUENCIA)
LayoutLM        : token → embedding(texto) + embedding(x, y en la PÁGINA)

     la caja delimitadora del OCR deja de tirarse a la basura
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('layoutlm', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué cadena produce el OCR de un documento de dos columnas?
2. ¿Cuántos campos empareja bien la regla «el siguiente token»?
3. ¿Y usando la posición?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('layoutlm', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('layoutlm', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La regla textual «el siguiente token» acierta **2 de 4** claves. Usando la posición —misma banda vertical, el valor más cercano a la derecha— acierta **4 de 4**. Los fallos no eran de lectura: el texto estaba bien reconocido y aun así el emparejado era incorrecto.


## 10. Comentario pedagógico

La información que faltaba eran **las coordenadas**, no más texto ni un modelo mayor. Esa es la tesis: la caja delimitadora que el OCR ya calcula y que las tuberías tiraban a la basura es la señal que resuelve el problema.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que el problema de la extracción de campos es de calidad de OCR.


In [ ]:
print('Con OCR perfecto, el emparejado por orden de lectura sigue fallando.')
print('El problema no es leer: es saber que va con que.')
print('Mejorar el OCR no mueve esa cifra.')

## 12. Corrección

Los dos criterios de emparejado, campo a campo:


In [ ]:
r = run_paper_lab('layoutlm', seed=3)['result']
print('orden del OCR:', r['orden_de_lectura_del_ocr'])
print('solo texto:', r['aciertos_solo_texto'])
for fila in r['emparejado_solo_con_texto']:
    print('  ', fila)
print('con posicion:', r['aciertos_con_posicion'])

## 13. Desafío guiado

Explica por qué la regla de posición del motor está escrita a mano y qué cambia cuando en su lugar se preentrena sobre millones de documentos.


In [ ]:
r = run_paper_lab('layoutlm', seed=3)['result']
show(r)

## 14. Desafío autónomo

Pasa un formulario real por un OCR que devuelva cajas. Comprueba cuántos pares clave-valor resuelve el orden de lectura y cuántos la posición.


## 15. Evidencia de aprendizaje

Guarda las dos cifras y los casos donde ninguna de las dos reglas funciona.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P125_layoutlm/README.md) · evaluación formal: [`assessments/papers/P125_layoutlm.md`](../../assessments/papers/P125_layoutlm.md)


## 16. Cierre

Si la posición basta, ¿hace falta el OCR? Esa pregunta es P126.


## 17. Conexión con el siguiente hito

- P126

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
